In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ── ADAPT-BIO Demo Notebook ───────────────────────────────────
# Author: Kritika | Manipal University Jaipur | April 2026
# GitHub: github.com/Kritika11052005/adapt-bio
# ─────────────────────────────────────────────────────────────

!git clone https://{GITHUB_TOKEN}@github.com/Kritika11052005/adapt-bio.git
%cd adapt-bio
!pip install torch transformers -q

import torch
import torch.nn as nn
import sys
sys.path.insert(0, '/kaggle/working/adapt-bio')

print("✅ ADAPT-BIO ready!")
print("   Biologically Inspired Sparse Attention")
print("   3 bio-signals: Slime Mold + Octopus RNA + Starling")

Cloning into 'adapt-bio'...
remote: Enumerating objects: 418, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 418 (delta 9), reused 32 (delta 6), pack-reused 381 (from 1)
Receiving objects: 100% (418/418), 4.18 MiB | 35.41 MiB/s, done.
Resolving deltas: 100% (168/168), done.
/kaggle/working/adapt-bio
✅ ADAPT-BIO ready!
   Biologically Inspired Sparse Attention
   3 bio-signals: Slime Mold + Octopus RNA + Starling


In [3]:
import torch
import torch.nn as nn
import numpy as np
import sys
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer

sys.path.insert(0, '/kaggle/working/adapt-bio')
from src.adapt_bio.models.adapt_bio_transformer import ADAPTBIOTransformer

# ── Fair Dense Baseline ────────────────────────────────────────────────────────
class FairDenseTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, seq_len, dropout=0.3):
        super().__init__()
        self.embed    = nn.Embedding(vocab_size, d_model)
        self.pos_enc  = nn.Embedding(seq_len, d_model)
        self.drop_emb = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=d_model * 4, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x, step=None):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        x = self.drop_emb(self.embed(x) + self.pos_enc(pos))
        x = self.transformer(x)
        return self.head(x)

# ── Config ─────────────────────────────────────────────────────────────────────
SEEDS      = [42, 123, 7]
STEPS      = 5000
SEQ_LEN    = 128
BATCH_SIZE = 32
D_MODEL    = 128
NUM_HEADS  = 4
NUM_LAYERS = 2
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Tokenizer ──────────────────────────────────────────────────────────────────
tokenizer  = AutoTokenizer.from_pretrained("bert-base-uncased")
VOCAB_SIZE = tokenizer.vocab_size   # 30522
print(f"Vocab size: {VOCAB_SIZE}")

# ── Dataset — continuous chunking, NO padding artifacts ───────────────────────
class WikiChunkDataset(Dataset):
    def __init__(self, split):
        data  = load_dataset("wikitext", "wikitext-2-raw-v1", split=split)
        text  = " ".join([x for x in data["text"] if len(x.strip()) > 0])
        ids   = tokenizer.encode(text, add_special_tokens=False)
        n     = (len(ids) // (SEQ_LEN + 1)) * (SEQ_LEN + 1)
        self.chunks = torch.tensor(ids[:n], dtype=torch.long).view(-1, SEQ_LEN + 1)
        print(f"  [{split}] chunks: {len(self.chunks)}")
    def __len__(self):
        return len(self.chunks)
    def __getitem__(self, i):
        return self.chunks[i]

print("Loading datasets...")
train_dataset = WikiChunkDataset("train")
val_dataset   = WikiChunkDataset("validation")
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# ── Evaluate ───────────────────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    criterion    = nn.CrossEntropyLoss()
    total_loss   = 0.0
    total_tokens = 0
    with torch.no_grad():
        for batch in loader:
            batch  = batch.to(DEVICE)
            inp, tgt = batch[:, :-1], batch[:, 1:]   # (B,128), (B,128)
            logits = model(inp, step=999999)
            loss   = criterion(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
            total_loss   += loss.item() * tgt.numel()
            total_tokens += tgt.numel()
    return np.exp(total_loss / total_tokens)

# ── Train + Eval ───────────────────────────────────────────────────────────────
def train_and_eval(ModelClass, seed, **model_kwargs):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model     = ModelClass(**model_kwargs).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()
    model.train()
    step        = 0
    loader_iter = iter(train_loader)
    while step < STEPS:
        try:
            batch = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader)
            batch = next(loader_iter)
        batch    = batch.to(DEVICE)
        inp, tgt = batch[:, :-1], batch[:, 1:]
        logits   = model(inp, step=step)
        loss     = criterion(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        step += 1
        if step % 500 == 0:
            print(f"    step {step}/{STEPS}  loss={loss.item():.4f}")
    return evaluate(model, val_loader)

# ── 3-Seed Loop ────────────────────────────────────────────────────────────────
results = {"dense": [], "adapt_bio": []}

for seed in SEEDS:
    print(f"\n{'='*50}")
    print(f"Seed {seed} — Dense")
    print(f"{'='*50}")
    ppl = train_and_eval(
        FairDenseTransformer, seed,
        vocab_size=VOCAB_SIZE, d_model=D_MODEL,
        num_heads=NUM_HEADS, num_layers=NUM_LAYERS, seq_len=SEQ_LEN
    )
    results["dense"].append(ppl)
    print(f"  ✓ Dense PPL = {ppl:.4f}")

    print(f"\n{'='*50}")
    print(f"Seed {seed} — ADAPT-BIO")
    print(f"{'='*50}")
    ppl = train_and_eval(
        ADAPTBIOTransformer, seed,
        vocab_size=VOCAB_SIZE, d_model=D_MODEL,
        num_heads=NUM_HEADS, num_layers=NUM_LAYERS, seq_len=SEQ_LEN,
        k=7, rna_update_interval=500
    )
    results["adapt_bio"].append(ppl)
    print(f"  ✓ ADAPT-BIO PPL = {ppl:.4f}")

# ── Final Summary ──────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  FINAL RESULTS — WikiText-2, 3 seeds")
print("="*55)
for name, ppls in results.items():
    mean, std = np.mean(ppls), np.std(ppls)
    print(f"  {name:12s}: PPL = {mean:.2f} ± {std:.2f}   runs: {[f'{p:.2f}' for p in ppls]}")
print("="*55)

ModuleNotFoundError: No module named 'src'